# 3_projects/youtube: A YouTube transcript server

**This server offers one tool, `get_transcript`: Give it a YouTube link, and it returns the video's captions as plain text.** A host like Claude Desktop can then summarize a video, or answer questions about it, without you watching it. The server is based on a tutorial and rewritten in this repo's style.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 host (Claude, or client.py)   server.py                  YouTube
 ---------------------------   ---------                  -------
 get_transcript(url) --stdio-&gt; extract_video_id(url)
                               api.fetch(video_id) -----&gt; captions
 the transcript text  &lt;------- TextFormatter() &lt;--------- snippets
</pre>

Read `../../1_crash-course/2-server-stdio-http-sse/client-stdio.ipynb` first. The connection is the same.

## 1. The library that fetches the captions

`youtube-transcript-api` fetches the captions YouTube already has for a video: The uploader's own, or YouTube's automatic ones. It needs no API key and doesn't download the video. `YouTubeTranscriptApi()` creates the fetcher, and `.fetch(video_id)` returns a `FetchedTranscript`:

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi

api = YouTubeTranscriptApi()
transcript = api.fetch("dQw4w9WgXcQ")

print(type(transcript))
print(transcript.language)
print(transcript.is_generated)
print(len(transcript.snippets))
print(transcript.snippets[1])

<class 'youtube_transcript_api._transcripts.FetchedTranscript'>
English
False
61
FetchedTranscriptSnippet(text="♪ We're no strangers to love ♪", start=18.64, duration=3.24)


- **`language` and `is_generated`:** English, written by the uploader (`False` means not auto-generated).
- **`snippets`:** The captions as a list of short pieces. Each one has its `text`, its `start` time in seconds, and how long it stays on screen (`duration`).

## 2. From snippets to plain text

`TextFormatter` joins the snippets' text into one string, one snippet per line, and drops the times:

In [2]:
from youtube_transcript_api.formatters import TextFormatter

text = TextFormatter().format_transcript(transcript)
print(len(text))
print(text[:150])

2089
[♪♪♪]
♪ We're no strangers to love ♪
♪ You know the rules
and so do I ♪
♪ A full commitment's
what I'm thinking of ♪
♪ You wouldn't get this
from any 


This string is what the tool returns, and what the model reads.

## 3. Finding the video ID

People paste links in many shapes, but the library needs the bare 11-character video ID. `extract_video_id` finds it with one regular expression:

<div style="font-size: 0.85em">

```python
r"(?:v=|youtu\.be/|embed/|shorts/)([0-9A-Za-z_-]{11})"
```

</div>

- **`(?:v=|youtu\.be/|embed/|shorts/)`:** What comes just before the ID. `(?: ... )` groups the choices without keeping them, and `|` separates them. `\.` means a real dot, because a plain `.` matches any character.
- **`([0-9A-Za-z_-]{11})`:** The ID itself, which is kept: Exactly 11 characters, each a digit, a letter, `_` or `-`. Every YouTube video ID has this shape.
- **`re.search` and `match.group(1)`:** Find the pattern anywhere in the string, and return the kept part. If nothing matches, the function returns the input as it is, assuming it's already a bare ID.

In [3]:
from server import extract_video_id

links = [
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://youtu.be/dQw4w9WgXcQ?t=42",
    "https://www.youtube.com/shorts/dQw4w9WgXcQ",
    "https://www.youtube.com/watch?list=PL1&v=dQw4w9WgXcQ",
    "dQw4w9WgXcQ",
]
for link in links:
    print(extract_video_id(link))

dQw4w9WgXcQ
dQw4w9WgXcQ
dQw4w9WgXcQ
dQw4w9WgXcQ
dQw4w9WgXcQ


## 4. The server

<div style="font-size: 0.85em">

```python
import re

from mcp.server.fastmcp import FastMCP
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter

mcp = FastMCP("YouTube")
api = YouTubeTranscriptApi()


def extract_video_id(url_or_id):
    match = re.search(r"(?:v=|youtu\.be/|embed/|shorts/)([0-9A-Za-z_-]{11})", url_or_id)
    if match:
        return match.group(1)
    return url_or_id


@mcp.tool()
def get_transcript(video_url_or_id: str) -> str:
    """Get the transcript of a YouTube video as plain text, from its URL or its 11-character video ID."""
    video_id = extract_video_id(video_url_or_id)
    transcript = api.fetch(video_id)
    return TextFormatter().format_transcript(transcript)


if __name__ == "__main__":
    mcp.run(transport="stdio")
```

</div>

- **The docstring:** It becomes the tool's description, so it tells the model that a URL or an ID both work.
- **No `try/except`:** If YouTube has no captions for a video, `api.fetch` raises an error, and FastMCP sends it back as a result with `isError: true`.
- **`api` is created once:** At startup, not on every call.

## 5. Calling it over MCP

The same connection as in Part 2, with the tool's one argument, `video_url_or_id`:

In [4]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command="python", args=["server.py"])

with open("server_log.txt", "w") as log:
    async with stdio_client(server, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("get_transcript", {"video_url_or_id": "https://youtu.be/dQw4w9WgXcQ"})

transcript_text = result.content[0].text
print(result.isError)
print(len(transcript_text))
print(transcript_text[:150])

False
2089
[♪♪♪]
♪ We're no strangers to love ♪
♪ You know the rules
and so do I ♪
♪ A full commitment's
what I'm thinking of ♪
♪ You wouldn't get this
from any 


The same 2,089 characters as in section 2, now fetched through the MCP server. `client.py` does the same as a script.

## 6. Using it from Claude Code or Claude Desktop

These steps aren't run here. Both hosts start the server from their own folder, so use full paths.

**Claude Code:** `claude mcp add <name> -- <command> [args...]` registers a stdio server, and `claude mcp list` shows it with a health check:

<div style="font-size: 0.85em">

```bash
claude mcp add youtube -- C:\Users\ashut\anaconda3\envs\MCP_env\python.exe C:\Users\ashut\ML_Practice\MCP\3_projects\youtube\server.py
claude mcp list
```

</div>

**Claude Desktop:** Add the server to `claude_desktop_config.json` (Settings, Developer, Edit Config), then restart it. The README shows the JSON.

## 7. Why the original version had a proxy

YouTube blocks transcript requests from many cloud providers' IP addresses. A server hosted in the cloud therefore needs a proxy, and the tutorial's version could use a paid one (Webshare). From your own computer, requests come from your home connection, so this version leaves the proxy out.

## Say it out loud

> "I found this server in a tutorial and rewrote it my way. It's a FastMCP server with one tool, `get_transcript`. It pulls the 11-character video ID out of any YouTube link with a regex, fetches the captions with `youtube-transcript-api`, and returns them as plain text, so a host like Claude can summarize a video. It runs over stdio, so Claude Desktop or Claude Code just start it as a child process. If I hosted it in the cloud, I'd need a proxy, because YouTube blocks cloud IP ranges."